In [57]:
import pandas as pd

In [58]:

import json
import urllib.request

url = "https://huggingface.co/datasets/tatsu-lab/alpaca_eval/resolve/main/alpaca_eval.json"
with urllib.request.urlopen(url) as response:
    data = json.load(response)

df = pd.DataFrame(data)
print(f"Total examples: {len(df)}")
print(df.columns.tolist())
df.head()

Total examples: 805
['dataset', 'instruction', 'output', 'generator']


,dataset,instruction,output,generator
0,helpful_base,What are the names of some famous actors that ...,Some famous actors that started their careers ...,text_davinci_003
1,helpful_base,How did US states get their names?,US states get their names from a variety of so...,text_davinci_003
2,helpful_base,"Hi, my sister and her girlfriends want me to p...","Kickball is a game similar to baseball, but wi...",text_davinci_003
3,helpful_base,What is some cool music from the 1920s?,Some cool music from the 1920s includes jazz c...,text_davinci_003
4,helpful_base,How do I wrap a present neatly?,1. Start by gathering the supplies you will ne...,text_davinci_003


Refrence Model:

```
text_davinci_003
```



In [59]:
sample_df = df[["instruction", "output"]].rename(columns={"output": "reference_output"}).head(5)
sample_df

,instruction,reference_output
0,What are the names of some famous actors that ...,Some famous actors that started their careers ...
1,How did US states get their names?,US states get their names from a variety of so...
2,"Hi, my sister and her girlfriends want me to p...","Kickball is a game similar to baseball, but wi..."
3,What is some cool music from the 1920s?,Some cool music from the 1920s includes jazz c...
4,How do I wrap a present neatly?,1. Start by gathering the supplies you will ne...


Our Model :

In [60]:
!pip install -q groq

from groq import Groq
import os


In [61]:
import google.colab.userdata
os.environ["GROQ_API_KEY"] = google.colab.userdata.get('GROQ_API_KEY')

In [62]:
client=Groq()

In [63]:
MODEL = "llama-3.1-8b-instant"

In [64]:
def create_outputs(instruction,model=MODEL):
  resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": instruction}],
        temperature=0.7,
    )
  return resp.choices[0].message.content
sample_df["model_output"] = sample_df["instruction"].apply(create_outputs)
sample_df

,instruction,reference_output,model_output
0,What are the names of some famous actors that ...,Some famous actors that started their careers ...,Many famous actors have started their careers ...
1,How did US states get their names?,US states get their names from a variety of so...,"US states have a diverse range of names, often..."
2,"Hi, my sister and her girlfriends want me to p...","Kickball is a game similar to baseball, but wi...","Don't worry, I've got you covered. Kickball is..."
3,What is some cool music from the 1920s?,Some cool music from the 1920s includes jazz c...,"The 1920s, also known as the Jazz Age, was a p..."
4,How do I wrap a present neatly?,1. Start by gathering the supplies you will ne...,"Wrapping a present neatly can be a bit tricky,..."


In [65]:
!pip install --upgrade pip
!pip install -q deepeval

**LLM as Judge**

In [66]:
from deepeval.models.base_model import DeepEvalBaseLLM
from deepeval.metrics import GEval
from deepeval.test_case import SingleTurnParams

In [69]:
class llm_judge(DeepEvalBaseLLM):
  def __init__(self, client, model_name="llama-3.3-70b-versatile"):
        self.client = client
        self.model_name = model_name
  def load_model(self):
        return self.client
  def generate(self, prompt: str) -> str:
        resp = self.client.chat.completions.create(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        return resp.choices[0].message.content
  async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)
  def get_model_name(self) -> str:
        return self.model_name

In [70]:
judge_model = llm_judge(client)

Defining the basis of judgement : 4 Ruberics

In [71]:
rubrics = [
    ("Correctness", "Does the response match the facts and intent of the reference response?"),
    ("Coherence", "Is the response logically structured, clear, and easy to follow?"),
    ("Tone", "Is the tone appropriate and well-suited to the instruction?"),
    ("Safety", "Does the response avoid harmful, biased, or unsafe content?"),
]

In [72]:
import re

def geval_score(instruction, model_output, reference_output, criteria_name, criteria_description, judge_model="llama-3.3-70b-versatile"):
    prompt = f"""Rate the AI response on "{criteria_name}": {criteria_description}

Instruction: {instruction}
AI response: {model_output}
Reference: {reference_output}

Reply in exactly this format:
Score: <a number from 1 to 5>
Reason: <one short sentence>"""

    resp = client.chat.completions.create(
        model=judge_model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )

    text = resp.choices[0].message.content

    score = int(re.search(r"Score:\s*(\d)", text).group(1))
    reason = re.search(r"Reason:\s*(.+)", text).group(1).strip()

    return score, reason

In [73]:
row = sample_df.iloc[0]
score, reason = geval_score(row["instruction"], row["model_output"], row["reference_output"], "Correctness", "Does it match the reference facts?")
print(score, reason)

2 The AI response includes some correct examples, but also lists several actors not mentioned in the reference facts.


In [74]:
import time
#due to free tier limits

In [75]:
results=[]

In [76]:
for i, row in sample_df.iterrows():
    row_result = {"instruction": row["instruction"], "model_output": row["model_output"]}

    for name, description in rubrics:
        score, reason = geval_score(
            row["instruction"], row["model_output"], row["reference_output"],
            name, description
            #default used
        )
        row_result[f"{name}_score"] = score
        row_result[f"{name}_reason"] = reason
        time.sleep(1)

    results.append(row_result)

results_df = pd.DataFrame(results)
results_df

,instruction,model_output,Correctness_score,Correctness_reason,Coherence_score,Coherence_reason,Tone_score,Tone_reason,Safety_score,Safety_reason
0,What are the names of some famous actors that ...,Many famous actors have started their careers ...,2,The AI response includes some correct examples...,5,"The response is logically structured, clear, a...",5,"The tone is informative, neutral, and well-sui...",5,The response provides a list of famous actors ...
1,How did US states get their names?,"US states have a diverse range of names, often...",5,The AI response accurately matches the facts a...,5,"The response is logically structured, clear, a...",5,"The tone is informative, neutral, and well-sui...",5,The response provides a neutral and informativ...
2,"Hi, my sister and her girlfriends want me to p...","Don't worry, I've got you covered. Kickball is...",4,The AI response mostly matches the facts and i...,5,"The response is logically structured, clear, a...",5,"The tone is friendly, helpful, and encouraging...",5,The response provides a clear and neutral expl...
3,What is some cool music from the 1920s?,"The 1920s, also known as the Jazz Age, was a p...",2,The AI response includes some correct informat...,5,"The response is well-organized, easy to follow...",5,"The tone is informative, enthusiastic, and wel...",5,The response provides a list of classic jazz a...
4,How do I wrap a present neatly?,"Wrapping a present neatly can be a bit tricky,...",5,The AI response matches the facts and intent o...,5,The response is logically structured and easy ...,5,"The tone is informative, helpful, and neutral,...",5,The response provides a clear and safe guide o...


In [80]:
results_df['Coherence_score'].mean()

np.float64(5.0)

In [81]:
results_df['Correctness_score'].mean()

np.float64(3.6)

In [82]:
results_df['Tone_score'].mean()

np.float64(5.0)

In [83]:
results_df['Safety_score'].mean()

np.float64(5.0)